<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/lectures_supplementary_code_samples/autodiff_mermaid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automatic differentiation: computation graphs

This notebook requests diagrams from the [Autodiff Visualizer Hugging Face Space](https://huggingface.co/spaces/simon-clmtd/autodiff-visualizer). The Space evaluates each expression and generates the graph; the notebook displays the returned PNG. An internet connection is required.

Read solid arrows upward for the forward values and dashed arrows downward for gradients. Pink nodes show the local derivative at each operation. When an input feeds several paths, its gradient combines the contributions from those paths.


In [ ]:
%pip install gradio_client

In [ ]:
from gradio_client import Client
from IPython.display import Image, display

client = Client("simon-clmtd/autodiff-visualizer")

def show_graph(expr, vals):
    png_path = client.predict(
        expr=expr,
        vals=vals,
        mode="Forward & Backward",
        notation="∂(...) / ∂x [Mathbook]",
        api_name="/generate_png",
    )
    display(Image(filename=png_path))

## 1. Multiplication: $o = x y$

The local derivatives are $\partial o/\partial x = y$ and $\partial o/\partial y = x$. At $x=2$ and $y=3$, the output is $6$ and the input gradients are $3$ and $2$.


In [ ]:
show_graph("x * y", "x=2, y=3")

## 2. A composite graph: $o = x^2 + y$

The gradient passes through the addition and then the squaring operation. At $x=3$ and $y=4$, $o=13$, $\partial o/\partial x=6$, and $\partial o/\partial y=1$.


In [ ]:
show_graph("(x * x) + y", "x=3, y=4")

## 3. Multiple paths: $z = x^2 y + (y + 2)$

Here $y$ influences the output through two paths. At $x=3$ and $y=4$, $z=42$, $\partial z/\partial x=24$, and $\partial z/\partial y=10$.


In [ ]:
show_graph("(x * x) * y + (y + 2)", "x=3, y=4")

## 4. Nested expression: $L = ((x_1+x_2)x_3)^2$

At $x_1=1$, $x_2=2$, and $x_3=3$, the output is $81$. Each input gradient is $54$.


In [ ]:
show_graph("((x_1 + x_2) * x_3)**2", "x_1=1, x_2=2, x_3=3")

## 5. Step-by-step walkthrough

The Space can also return an HTML trace. This example revisits $z=x^2y+(y+2)$ so you can compare each step with the graph above.

👉 **[Click here to open the Autodiff Visualizer Space directly in a new window](https://huggingface.co/spaces/simon-clmtd/autodiff-visualizer)** to interact with the fully dynamic step-by-step walkthrough with all button-hover and focus features working natively.

In [ ]:
import base64
from IPython.display import HTML, display
from google.colab import files, output

walkthrough = client.predict(
    "(x * x) * y + (y + 2)",
    "x=3, y=4",
    "Forward & Backward",
    "∂(...) / ∂x [Mathbook]",
    api_name="/generate_walkthrough",
)

# Self-contained JS payload to enable local folding/unfolding and adjoint focus interactions
interactive_script = """
<script>
document.addEventListener('DOMContentLoaded', () => {
    // 1. Fold / Unfold All Helper (if buttons exist)
    const toggleButtons = document.querySelectorAll('button[id*="toggle"], button[id*="fold"]');
    toggleButtons.forEach(btn => {
        btn.addEventListener('click', () => {
            const details = document.querySelectorAll('details');
            const shouldCollapse = btn.textContent.toLowerCase().includes('fold') || btn.textContent.toLowerCase().includes('collapse');
            details.forEach(d => d.open = !shouldCollapse);
        });
    });

    // 2. Focus interaction on target nodes / adjoints
    const adjoints = document.querySelectorAll('.tv-adjoint, [class*="adjoint"], [class*="focus"]');
    adjoints.forEach(adj => {
        adj.style.cursor = 'pointer';
        adj.addEventListener('click', (e) => {
            e.stopPropagation();
            const targetId = adj.getAttribute('data-target') || adj.id;
            if (!targetId) return;

            const isActive = adj.classList.contains('focused');

            // Clear previous highlights
            document.querySelectorAll('.focused').forEach(el => el.classList.remove('focused'));
            document.querySelectorAll('.dimmed').forEach(el => el.classList.remove('dimmed'));

            if (!isActive) {
                adj.classList.add('focused');
                // Highlight related path components
                const related = document.querySelectorAll(`[data-related*="${targetId}"], [id*="${targetId}"]`);
                const allInteractive = document.querySelectorAll('.tv-adjoint, .tv-op, .tv-node');
                allInteractive.forEach(node => {
                    if (!node.classList.contains('focused') && ![...related].includes(node)) {
                        node.classList.add('dimmed');
                    }
                });
            }
        });
    });
});
</script>
<style>
.focused { outline: 2px solid #ff9d00 !important; background-color: rgba(255, 157, 0, 0.1) !important; transition: all 0.2s; }
.dimmed { opacity: 0.4 !important; transition: all 0.2s; }
</style>
"""

# Inject script at the end of the HTML structure
if "</body>" in walkthrough:
    interactive_walkthrough = walkthrough.replace("</body>", f"{interactive_script}</body>")
else:
    interactive_walkthrough = walkthrough + interactive_script

# Save the enhanced, fully interactive HTML locally
filename = "walkthrough.html"
with open(filename, "w", encoding="utf-8") as f:
    f.write(interactive_walkthrough)

# Register helper function to trigger fallback download
def trigger_download():
    files.download(filename)

output.register_callback('notebook.download_file', trigger_download)

# Create a direct base64 HTML download link
b64_data = base64.b64encode(interactive_walkthrough.encode('utf-8')).decode('utf-8')
download_href = f"data:text/html;base64,{b64_data}"

# Present UI controls
fallback_ui = f"""
<div style="padding: 20px; border: 1px solid #d8dbe0; border-radius: 10px; background-color: #fcfcfb; font-family: sans-serif;">
    <h3 style="margin-top: 0; color: #202124;">Interactive Trace Walkthrough</h3>
    <p style="color: #5f6368; font-size: 14px; margin-bottom: 20px;">
        We have injected direct browser event handling code into the HTML payload to enable fully native folding and node highlighting in your local browser tab.
    </p>
    <div style="display: flex; gap: 15px; flex-wrap: wrap;">
        <a href="{download_href}" download="walkthrough.html"
           style="display: inline-block; padding: 10px 18px; background-color: #1a73e8; color: white; text-decoration: none; border-radius: 5px; font-weight: bold; font-size: 14px;">
            📥 Download Interactive HTML File
        </a>
        <a href="https://huggingface.co/spaces/simon-clmtd/autodiff-visualizer" target="_blank"
           style="display: inline-block; padding: 10px 18px; background-color: #ff9d00; color: white; text-decoration: none; border-radius: 5px; font-weight: bold; font-size: 14px;">
            🤗 Open on Hugging Face Space
        </a>
    </div>
</div>
"""

display(HTML(fallback_ui))

Try changing an expression or its values above. The Space also provides `/generate_svg`, `/generate_pdf`, and `/generate_mermaid` endpoints for other forms of the same result.
